# Indsætter renset data

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
file_path3 = '/Users/cerinavonbruhn/Desktop/glathida_rgis_edited1.csv'
latent_space_data_merged32 = pd.read_csv(file_path3)

In [ ]:
latent_space_data_merged32

In [ ]:
latent_space_data = latent_space_data.drop(['RGIId'], axis = 1)
latent_space_data = latent_space_data.drop(['Form_x'], axis = 1)
latent_space_data = latent_space_data.drop(['POLITICAL_UNIT'], axis = 1)
latent_space_data = latent_space_data.drop(['REMARKS'], axis = 1)
latent_space_data = latent_space_data.drop(['GlaThiDa_ID'], axis = 1)
latent_space_data = latent_space_data.drop(['POINT_ID'], axis = 1)

In [ ]:
print(latent_space_data.columns)

In [ ]:
# Beregn gennemsnittet indenfor hver RGIId
mean_values = latent_space_data_merged32.groupby('RGIId')[['POINT_LAT','POINT_LON']].transform('mean')

# Opret de lokale koordinater ved at subtrahere gennemsnittet
latent_space_data_merged32['local_longitude'] = latent_space_data_merged32['POINT_LON'] - mean_values['POINT_LON']
latent_space_data_merged32['local_latitude'] = latent_space_data_merged32['POINT_LAT'] - mean_values['POINT_LAT']

In [ ]:
# Beregn gennemsnittet indenfor hver RGIId
mean_values1 = latent_space_data_merged32.groupby('RGIId')[['POINT_LAT','POINT_LON']].transform('mean')

# Opret de lokale koordinater ved at subtrahere gennemsnittet
latent_space_data_merged32['local_longitude'] = latent_space_data_merged32['POINT_LON'] - mean_values1['POINT_LON']
latent_space_data_merged32['local_latitude'] = latent_space_data_merged32['POINT_LAT'] - mean_values1['POINT_LAT']

In [ ]:
latent_space_data_merged32 = latent_space_data_merged32.drop(['RGIId'], axis = 1)
latent_space_data_merged32 = latent_space_data_merged32.drop(['Form_x'], axis = 1)
latent_space_data_merged32 = latent_space_data_merged32.drop(['POLITICAL_UNIT'], axis = 1)
latent_space_data_merged32 = latent_space_data_merged32.drop(['REMARKS'], axis = 1)
latent_space_data_merged32 = latent_space_data_merged32.drop(['GlaThiDa_ID'], axis = 1)
latent_space_data_merged32 = latent_space_data_merged32.drop(['POINT_ID'], axis = 1)

## Gradient Boosting Decision Trees (GBDT)

In [ ]:
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
# Adskil features og target
X = latent_space_data_merged32.drop(columns=['THICKNESS'])
y = latent_space_data_merged32['THICKNESS']

# Opdel data i trænings- og test-sæt
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Definér en Optuna objective-funktion
def objective(trial):
    param = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'lambda_l1': trial.suggest_loguniform('lambda_l1', 1e-2, 10.0),
        'lambda_l2': trial.suggest_loguniform('lambda_l2', 1e-2, 10.0),
        'num_leaves': trial.suggest_int('num_leaves', 2, 256),
        'feature_fraction': trial.suggest_uniform('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-2, 0.1)
    }

    # Brug krydstræning til evaluering
    cv = KFold(n_splits=2, shuffle=True, random_state=42)
    cv_results = lgb.cv(param, lgb.Dataset(X_train, label=y_train), nfold=cv.get_n_splits(), metrics='mae', seed=42, stratified=False)
    
    # Print tilgængelige nøgler i cv_results
    print(f'Available keys in cv_results: {cv_results.keys()}')
    
    # Kontrollér hvilke nøgler der er tilgængelige
    if 'l1-mean' in cv_results:
        mean_mae = cv_results['l1-mean'][-1]
    elif 'valid l1-mean' in cv_results:
        mean_mae = cv_results['valid l1-mean'][-1]
    else:
        raise KeyError("Expected keys 'l1-mean' or 'valid l1-mean' not found in cv_results")
    
    return mean_mae

# Start Optuna-optimering
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)

In [ ]:
# Udskriv de bedste hyperparametre
print('Number of finished trials:', len(study.trials))
print('Best trial:')
trial = study.best_trial

print('  RMSE:', trial.value)
print('  Best hyperparameters:', trial.params)

In [ ]:
# Træn den endelige model med de bedste hyperparametre
best_params = trial.params
dtrain = lgb.Dataset(X_train, label=y_train)
dtest = lgb.Dataset(X_test, label=y_test)
final_gbm = lgb.train(best_params, dtrain, num_boost_round=1000)

In [ ]:
# Evaluer den endelige model
preds = final_gbm.predict(X_test)
final_mae = mean_absolute_error(y_test, preds)
print('Final MAE:', final_mae)

## XGBoost

In [ ]:
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
# Definér en Optuna
def objective(trial):
    param = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'booster': 'gbtree',
        'lambda': trial.suggest_loguniform('lambda', 1e-3, 10.0),
        'alpha': trial.suggest_loguniform('alpha', 1e-3, 10.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.4, 1.0),
        'subsample': trial.suggest_uniform('subsample', 0.4, 1.0),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 1, 20),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_loguniform('gamma', 1e-3, 10.0)
    }

    # Brug krydstræning til evaluering
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_results = xgb.cv(param, xgb.DMatrix(X_train, label=y_train), num_boost_round=100, nfold=cv.get_n_splits(), metrics='mae', seed=42, stratified=False, early_stopping_rounds=100)
    
    # Print tilgængelige nøgler i cv_results
    print(f'Available keys in cv_results: {cv_results.keys()}')

    # Returner gennemsnittet af MAE fra krydstræningen
    mean_mae = cv_results['test-mae-mean'].iloc[-1]
    
    return mean_mae

In [ ]:
# Start Optuna-optimering
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)

In [ ]:
# Udskriv de bedste hyperparametre
print('Number of finished trials:', len(study.trials))
print('Best trial:')
trial = study.best_trial

print('  MAE:', trial.value)
print('  Best hyperparameters:', trial.params)

In [ ]:
# Træn den endelige model med de bedste hyperparametre
best_params = trial.params
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)
final_gbm = xgb.train(best_params, dtrain, num_boost_round=1000)

In [ ]:
# Evaluer den endelige model
preds = final_gbm.predict(dtest)
final_mae = mean_absolute_error(y_test, preds)
print('Final MAE:', final_mae)

In [ ]:
# Opret en DataFrame
data = {'preds': preds1, 'y': y}
df = pd.DataFrame(data)

# Gem DataFrame til en CSV-fil
df.to_csv('for_metadata_uden_latent.csv', index=False)

#print("CSV-filen er gemt som 'predictionstilmarcus.csv'")